# Plotting API Usage

This notebook demonstrates the stateless plotting helpers in `plotting/`. Each plotting function receives `df` and/or `trajectories`, then returns a Matplotlib figure and axes.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path.cwd().resolve()
if not (PROJECT / "plotting").exists():
    PROJECT = PROJECT.parent.resolve()
sys.path.insert(0, str(PROJECT))

from plotting import (
    PLOT_COLORS,
    ordered_algos_in,
    ordered_dims_in,
    plot_algorithm_dimension_grid,
    plot_algorithms_for_dimension,
    plot_all_mean_curves_combined,
    plot_color_key,
    plot_dimensions_for_algorithm,
    plot_metric_bar,
    plot_metric_overview,
    plot_seed_variability_for_dimension,
    summary_table,
)


def show_figure(fig):
    display(fig)
    plt.close(fig)

## Toy Inputs

The real experiment notebook produces these two objects in memory:

- `df`: one row per run
- `trajectories`: keyed by `(algo, d, seed)` with per-step loss lists

In [ ]:
ALGOS = ["Muon", "Muon-Exact", "Shampoo", "Adam", "SGD"]
DIMS = [20, 30]
SEEDS = range(3)
ITERS = 60

rate = {"Muon": 0.075, "Muon-Exact": 0.082, "Shampoo": 0.06, "Adam": 0.052, "SGD": 0.038}
scale = {"Muon": 0.75, "Muon-Exact": 0.72, "Shampoo": 0.9, "Adam": 1.0, "SGD": 1.15}

rows = []
trajectories = {}
rng = np.random.default_rng(0)
for algo in ALGOS:
    for d in DIMS:
        for seed in SEEDS:
            steps = np.arange(ITERS)
            noise = rng.normal(0.0, 0.01, size=ITERS)
            loss = scale[algo] * np.exp(-rate[algo] * steps * (20 / d)) + 0.01 + np.abs(noise)
            trajectories[(algo, d, seed)] = {"loss": loss.tolist(), "grad_norm": np.sqrt(loss).tolist()}
            rows.append({
                "problem": "ToyMatrixSensing",
                "algo": algo,
                "d": d,
                "seed": seed,
                "iters": ITERS,
                "actual_steps": ITERS,
                "stopped_early": False,
                "min_loss": float(loss.min()),
                "final_loss": float(loss[-1]),
                "time_s": float((d / 20) * scale[algo] * (1.0 + seed / 20)),
            })

df = pd.DataFrame(rows)
display(df.head())
display(summary_table(df))

## Shared Colors

In [ ]:
PLOT_COLORS

## Color Key

In [ ]:
fig, axes = plot_color_key(df)
show_figure(fig)

## Metric Overview

In [ ]:
fig, axes = plot_metric_overview(df)
show_figure(fig)

## Metric Bar

In [ ]:
fig, ax = plot_metric_bar(df, "actual_steps_mean", "Mean executed steps", "steps")
show_figure(fig)

## Same Dimension

In [ ]:
fig, ax = plot_algorithms_for_dimension(trajectories, d=20)
show_figure(fig)

## Same Algorithm

In [ ]:
fig, ax = plot_dimensions_for_algorithm(trajectories, algo="Muon")
show_figure(fig)

## All Mean Curves

In [ ]:
fig, ax = plot_all_mean_curves_combined(trajectories)
show_figure(fig)

## Algorithm-Dimension Grid

In [ ]:
fig, axes = plot_algorithm_dimension_grid(trajectories)
show_figure(fig)

## Seed Variability

In [ ]:
fig, ax = plot_seed_variability_for_dimension(trajectories, d=20)
show_figure(fig)